# PolyLLM Chairman Training on Kaggle

Run cells in order.

Before running:
1. Enable GPU in Kaggle notebook settings.
2. Add a Kaggle dataset containing `train.jsonl` and `val.jsonl`.
3. Optionally add a dataset with saved checkpoints (`checkpoint-*`) to resume.
4. Add Kaggle Secrets: `HF_TOKEN`, `HF_USERNAME`.

## Cell-by-cell actions

1. Cell 1: Run as-is.
2. Cell 2: Run as-is.
3. Cell 3: Run as-is.
4. Cell 4: Edit required. Add Kaggle Secrets HF_TOKEN and HF_USERNAME before running.
5. Cell 5: Edit required. Set DATASET_DIR to your Kaggle input dataset path before running.
6. Cell 6: Run as-is.
7. Cell 7: Run as-is.
8. Cell 8: Run as-is.
9. Cell 9: Run as-is.
10. Cell 10: Edit required only if needed. Set RESUME_INPUT_ROOT if your checkpoint dataset path differs; otherwise run as-is.
11. Cell 11: Run as-is.
12. Cell 12: Run as-is.
13. Cell 13: Run as-is.

In [ ]:
# Cell 1: Install packages
!pip -q install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip -q install --no-deps "trl>=0.11.0" "peft>=0.13.0" "accelerate>=0.34.0" "datasets>=3.0.0" huggingface_hub
!pip -q install modelscope

print("Packages installed")
print("If Kaggle asks for restart, restart the session and continue from Cell 2.")

In [ ]:
# Cell 2: Environment setup
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"

print("Environment variables applied")

In [ ]:
# Cell 3: GPU check
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("CUDA:", torch.version.cuda)

In [ ]:
# Cell 4: Hugging Face login from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
HF_USERNAME = secrets.get_secret("HF_USERNAME")

login(token=HF_TOKEN)
print("Logged in to Hugging Face as:", HF_USERNAME)

In [ ]:
# Cell 5: Dataset paths
import os

# Update this to your Kaggle input dataset path
DATASET_DIR = "/kaggle/input/polyllm-dataset"

TRAIN_PATH = f"{DATASET_DIR}/train.jsonl"
VAL_PATH = f"{DATASET_DIR}/val.jsonl"

assert os.path.exists(TRAIN_PATH), f"Missing {TRAIN_PATH}"
assert os.path.exists(VAL_PATH), f"Missing {VAL_PATH}"

print("Train:", TRAIN_PATH)
print("Val:", VAL_PATH)

In [ ]:
# Cell 6: Load and format dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": TRAIN_PATH,
    "validation": VAL_PATH,
})

print("Train examples:", len(dataset["train"]))
print("Val examples:", len(dataset["validation"]))

def format_example(example):
    text = (
        f"<s>[INST] <<SYS>>\n{example['instruction']}\n<</SYS>>\n\n"
        f"{example['input']} [/INST] "
        f"{example['output']} </s>"
    )
    return {"text": text}

dataset = dataset.map(
    format_example,
    remove_columns=["id", "domain", "instruction", "input", "output"],
)

print("Sample:", dataset["train"][0]["text"][:300])

In [ ]:
# Cell 7: Load base model in 4-bit
from unsloth import FastLanguageModel
import torch

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_SEQ_LEN = 4096

print("Loading", BASE_MODEL, "in 4-bit")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded. VRAM used:", f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 8: Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

In [ ]:
# Cell 9: Training arguments
from transformers import TrainingArguments
import os

WORK_DIR = "/kaggle/working/polyllm-chairman"
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
ADAPTER_DIR = f"{WORK_DIR}/adapter"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=11,
    logging_steps=10,
    eval_steps=50,
    save_steps=25,
    save_total_limit=10,
    eval_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    report_to="none",
    dataloader_pin_memory=False,
)

print("Training args configured")
print("Checkpoint dir:", CHECKPOINT_DIR)

In [ ]:
# Cell 10: Trainer and robust auto-resume
from trl import SFTTrainer
import glob
import os
import re

# If you know the exact checkpoint folder, set it here.
# Example: "/kaggle/input/checkpoint-200"
# Keep None to auto-detect.
RESUME_CHECKPOINT_PATH = None

# Can be either:
# - parent folder containing checkpoint-* dirs
# - direct checkpoint dir itself
RESUME_INPUT_ROOT = "/kaggle/input/checkpoint-200"

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

def checkpoint_step(path):
    match = re.search(r"checkpoint-(\d+)$", os.path.basename(path.rstrip("/")))
    return int(match.group(1)) if match else -1

def is_valid_checkpoint_dir(path):
    if not os.path.isdir(path):
        return False
    required = ["trainer_state.json", "training_args.bin"]
    return all(os.path.exists(os.path.join(path, f)) for f in required)

# Search roots include likely mount points
search_roots = [
    CHECKPOINT_DIR,
    RESUME_INPUT_ROOT,
    "/kaggle/input",
]

print("Checkpoint search roots:")
for root in search_roots:
    print(" -", root)

candidates = set()

# 1) Manual exact override takes highest priority
if RESUME_CHECKPOINT_PATH:
    if is_valid_checkpoint_dir(RESUME_CHECKPOINT_PATH):
        candidates.add(RESUME_CHECKPOINT_PATH)
    else:
        print("Warning: RESUME_CHECKPOINT_PATH is set but invalid:", RESUME_CHECKPOINT_PATH)

# 2) Discover checkpoints from roots
for root in search_roots:
    if not os.path.exists(root):
        continue

    # If root itself is a checkpoint dir (e.g. /kaggle/input/checkpoint-200)
    if os.path.basename(root).startswith("checkpoint-") and is_valid_checkpoint_dir(root):
        candidates.add(root)

    # If checkpoints are nested one level below root
    for ckpt in glob.glob(os.path.join(root, "checkpoint-*")):
        if is_valid_checkpoint_dir(ckpt):
            candidates.add(ckpt)

    # If dataset layout is /kaggle/input/<dataset>/checkpoint-*
    for ckpt in glob.glob(os.path.join(root, "*", "checkpoint-*")):
        if is_valid_checkpoint_dir(ckpt):
            candidates.add(ckpt)

sorted_checkpoints = sorted(candidates, key=checkpoint_step)

if sorted_checkpoints:
    print("Found checkpoints (oldest -> newest):")
    for ckpt in sorted_checkpoints:
        print(f" - step {checkpoint_step(ckpt)}: {ckpt}")
    resume_from = sorted_checkpoints[-1]
    print("Mode: RESUME", resume_from)
else:
    resume_from = None
    print("Found checkpoints: none")
    print("Mode: FRESH")

trainer.train(resume_from_checkpoint=resume_from)
print("Training complete")

In [ ]:
# Cell 11: Save adapter
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to:", ADAPTER_DIR)

In [ ]:
# Cell 12: Push adapter to Hugging Face Hub
repo_id = f"{HF_USERNAME}/polyllm-chairman"

model.push_to_hub(repo_id, token=HF_TOKEN)
tokenizer.push_to_hub(repo_id, token=HF_TOKEN)

print("Pushed to:", repo_id)

In [ ]:
# Cell 13: Quick inference test (Unsloth-compatible, cleaner output)
import warnings
import torch
from transformers import GenerationConfig
from unsloth import FastLanguageModel
from peft import PeftModel

# Silence known non-blocking deprecation warnings from current Transformers internals
warnings.filterwarnings(
    "ignore",
    message=r".*AttentionMaskConverter.*deprecated.*",
    category=FutureWarning,
    module=r"transformers\.modeling_attn_mask_utils",
)

# Reload base model through Unsloth so required runtime patches are present
inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)

# Load trained LoRA adapter
inference_model = PeftModel.from_pretrained(inference_model, ADAPTER_DIR)
FastLanguageModel.for_inference(inference_model)
inference_model.eval()

prompt = "<s>[INST] <<SYS>>You are a council chairman. Synthesize the best final answer. <</SYS>>\n\nQuestion: Why is the sky blue? [/INST]"

inputs = inference_tokenizer(prompt, return_tensors="pt").to("cuda")

# Use a dedicated generation config to avoid generation_config/kwargs deprecation warnings
gen_config = GenerationConfig.from_model_config(inference_model.config)
gen_config.max_new_tokens = 200
gen_config.do_sample = False
gen_config.pad_token_id = inference_tokenizer.eos_token_id
gen_config.eos_token_id = inference_tokenizer.eos_token_id
gen_config.max_length = None

with torch.no_grad():
    outputs = inference_model.generate(
        **inputs,
        generation_config=gen_config,
    )

# Cleaner decoded text (removes duplicated special tokens like <s>)
print(inference_tokenizer.decode(outputs[0], skip_special_tokens=True))